# 03 - Preprocesamiento

**TP2 - Modulo 2 - Clasificacion `smoking`**

Aca armamos la receta de transformacion de los datos pero sin aplicarla todavia. El ajuste ocurre en el notebook 04 usando solo los datos de entrenamiento, para que la validacion y la prediccion no contaminen nada.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import pandas as pd
import numpy as np
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

import utils
from utils import DATA_PROCESSED, MODELS, DROP_COLS, TARGET

df = pd.read_csv(DATA_PROCESSED / 'train_clean.csv')
X, y = utils.features_target(df)
print('Features disponibles (sin id/oral/target):', X.shape[1])
print(list(X.columns))

Features disponibles (sin id/oral/target): 32
['gender', 'age', 'height_cm', 'weight_kg', 'waist_cm', 'eyesight_left', 'eyesight_right', 'hearing_left', 'hearing_right', 'systolic', 'relaxation', 'fasting_blood_sugar', 'cholesterol', 'triglyceride', 'hdl', 'ldl', 'hemoglobin', 'urine_protein', 'serum_creatinine', 'ast', 'alt', 'gtp', 'dental_caries', 'tartar', 'bmi', 'waist_to_height', 'pulse_pressure', 'map', 'ast_alt_ratio', 'non_hdl', 'tg_hdl_ratio', 'liver_enzymes_sum']


## Clasificacion de columnas

- `gender` y `tartar` son texto -> las convertimos a numeros con one-hot encoding.
- El resto son numericas y pasan tal cual (incluyendo las binarias como `dental_caries`).

In [2]:
cat_cols = [c for c in ['gender', 'tartar'] if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

print('Categoricas:', cat_cols)
print('Numericas  :', num_cols)

Categoricas: ['gender', 'tartar']
Numericas  : ['age', 'height_cm', 'weight_kg', 'waist_cm', 'eyesight_left', 'eyesight_right', 'hearing_left', 'hearing_right', 'systolic', 'relaxation', 'fasting_blood_sugar', 'cholesterol', 'triglyceride', 'hdl', 'ldl', 'hemoglobin', 'urine_protein', 'serum_creatinine', 'ast', 'alt', 'gtp', 'dental_caries', 'bmi', 'waist_to_height', 'pulse_pressure', 'map', 'ast_alt_ratio', 'non_hdl', 'tg_hdl_ratio', 'liver_enzymes_sum']


## Receta de transformacion

Para las numericas usamos imputacion por mediana (por si aparecen nulos en prediccion). Para las categoricas imputacion por moda y one-hot. `handle_unknown='ignore'` es por si aparece algun valor nuevo en prediccion que no estaba en train.

In [3]:
cat_pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('ohe', OneHotEncoder(handle_unknown='ignore', drop='if_binary')),
])

num_pipe_arbol = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
])

prep_arbol = ColumnTransformer([
    ('num', num_pipe_arbol, num_cols),
    ('cat', cat_pipe, cat_cols),
], remainder='drop')

print('Preprocesador construido (SIN fittear).')

Preprocesador construido (SIN fittear).


## Guardado

Guardamos la receta sin ajustar para que el notebook 04 la use dentro del pipeline completo.

In [4]:
MODELS.mkdir(parents=True, exist_ok=True)
joblib.dump(prep_arbol, MODELS / 'preprocessor_arbol_unfitted.joblib')

feature_spec = {
    'target': TARGET,
    'drop_cols': DROP_COLS,
    'cat_cols': cat_cols,
    'num_cols': num_cols,
    'all_features': list(X.columns),
}
with open(DATA_PROCESSED / 'feature_spec.json', 'w') as f:
    json.dump(feature_spec, f, indent=2)

print('Guardados preprocesador y feature_spec.json')

Guardados preprocesador y feature_spec.json


## Decisiones tomadas

- Sacamos `id` y `oral` antes de modelar: el primero es solo un numero de fila, el segundo es constante.
- Imputamos con mediana para numericas y moda para categoricas, ajustando solo en train.
- No eliminamos outliers: XGBoost los tolera bien y no podriamos replicar la eliminacion en el set de prediccion de forma confiable.
- Todo el ajuste queda dentro del pipeline para que no haya forma de que informacion de validacion o prediccion se cuele en el entrenamiento.